# Day 14 — Project: Customer Churn Prediction Neural Network

## 1. Learning Objectives
- Synthesize `nn.Module`, `BCEWithLogitsLoss`, and `optim.Adam`.
- Handle a dataset with both categorical and numerical variables.
- Build a robust Train/Val/Test pipeline.
- Perform Error Analysis on the results using Precision and Recall.

## 2. Project Overview
You are hired by a Telecom company. They want to predict whether a customer will "Churn" (cancel their subscription). This is a **Binary Classification** problem. We will build a Neural Network to predict Churn.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

## Step 1: Create the Dataset
We will simulate a dataset of 1000 customers.
Features:
- `Age` (Numerical: 18 to 80)
- `MonthlyCharge` (Numerical: $20 to $120)
- `Is_International_Plan` (Categorical: 0 or 1)
- `Customer_Service_Calls` (Numerical: 0 to 10)

In [ ]:
torch.manual_seed(42)
num_samples = 1000

age = torch.randint(18, 80, (num_samples, 1)).float()
charge = torch.rand(num_samples, 1) * 100 + 20
intl_plan = torch.randint(0, 2, (num_samples, 1)).float()
calls = torch.randint(0, 10, (num_samples, 1)).float()

X_raw = torch.cat([age, charge, intl_plan, calls], dim=1)

# Simulate Churn logic: 
# High calls, high charge, and no intl plan highly increases churn.
churn_score = (calls * 1.5) + (charge * 0.05) - (intl_plan * 2) - (age * 0.01) + torch.randn(num_samples, 1) * 2
# If score > 6, they churn (1), else (0)
y = (churn_score > 6).float()

print("Total Churners:", y.sum().item(), "out of", num_samples)

## Step 2: Preprocessing
Neural networks need all inputs to be roughly on the same scale (mean 0, std 1). We normalize the numerical columns.

In [ ]:
# Normalize Age (idx 0), Charge (idx 1), Calls (idx 3)
X_norm = X_raw.clone()
for i in [0, 1, 3]:
    col_mean = X_norm[:, i].mean()
    col_std = X_norm[:, i].std()
    X_norm[:, i] = (X_norm[:, i] - col_mean) / col_std
    
# Split 80/20
split = int(0.8 * num_samples)
X_train, X_val = X_norm[:split], X_norm[split:]
y_train, y_val = y[:split], y[split:]

## Step 3: Define the Neural Network
**Your Task:** Define a network with `in_features=4` (our 4 columns). Add two hidden layers with ReLU. The output must be `out_features=1`.

In [ ]:
class ChurnModel(nn.Module):
    def __init__(self):
        super().__init__()
        # Define layers here
        self.fc1 = nn.Linear(4, 16)
        self.fc2 = nn.Linear(16, 8)
        self.fc3 = nn.Linear(8, 1)
        self.relu = nn.ReLU()
        
    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.relu(self.fc2(x))
        # Note: No Sigmoid here because we will use BCEWithLogitsLoss
        return self.fc3(x)

model = ChurnModel()
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

## Step 4: The Training Loop
We will run 100 epochs and evaluate on the Validation set.

In [ ]:
epochs = 100
train_losses, val_losses = [], []

for epoch in range(epochs):
    model.train()
    
    # Forward
    y_pred = model(X_train)
    loss = criterion(y_pred, y_train)
    
    # Backward
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    train_losses.append(loss.item())
    
    # Validation
    model.eval()
    with torch.no_grad():
        y_val_pred = model(X_val)
        v_loss = criterion(y_val_pred, y_val)
        val_losses.append(v_loss.item())
        
    if epoch % 20 == 0:
        print(f"Epoch {epoch} | Train Loss: {loss.item():.4f} | Val Loss: {v_loss.item():.4f}")

## Step 5: Evaluation & Error Analysis
Loss curves are great, but let's calculate Precision and Recall.

In [ ]:
model.eval()
with torch.no_grad():
    raw_logits = model(X_val)
    # Since we didn't use Sigmoid in the model, we must apply it now to get probabilities
    probs = torch.sigmoid(raw_logits)
    
    # Convert probabilities to 1 (Churn) or 0 (Stay) using 0.5 threshold
    predictions = (probs > 0.5).float()
    
    TP = ((predictions == 1) & (y_val == 1)).sum().float()
    FP = ((predictions == 1) & (y_val == 0)).sum().float()
    FN = ((predictions == 0) & (y_val == 1)).sum().float()
    
    precision = TP / (TP + FP + 1e-8)
    recall = TP / (TP + FN + 1e-8)
    
    print(f"Validation Precision: {precision.item():.2f}")
    print(f"Validation Recall: {recall.item():.2f}")

## Phase 2 Conclusion
You have successfully built an Object-Oriented PyTorch Neural Network for Classification! 

In **Phase 3 (Advanced Training)**, we will fix a major flaw in what we've done so far. Notice how we passed the *entire dataset* into the model at once (`model(X_train)`)? If your dataset has 10 million rows, your GPU memory will crash instantly. Next week, we introduce `Dataset` and `DataLoader` for **Mini-Batching**.